In [1]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

StatementMeta(, d7e4f1a3-25db-4950-a241-8f56f7ba241c, 3, Finished, Available, Finished, False)

In [2]:
# 1) Read tables from Lakehouse
positions = spark.table("dbo.positions")
trades = spark.table("dbo.trades")

StatementMeta(, d7e4f1a3-25db-4950-a241-8f56f7ba241c, 4, Finished, Available, Finished, False)

In [4]:
# 2) Normalize types
positions_typed = (
    positions
    .withColumn("position_date", F.to_date("position_date"))
    .withColumn("trader_id", F.col("trader_id").cast("int"))
    .withColumn("quantity", F.col("quantity").cast("double"))
    .withColumn("avg_price", F.col("avg_price").cast("double"))
    .withColumn("cost_basis_total", F.col("cost_basis_total").cast("double"))
)

trades_typed = (
    trades
    .withColumn("data_hora_ts", F.to_timestamp("data_hora"))
    .withColumn("trade_date", F.to_date("data_hora_ts"))
    .withColumn("comprador_id", F.col("comprador_id").cast("int"))
    .withColumn("vendedor_id", F.col("vendedor_id").cast("int"))
    .withColumn("quantidade", F.col("quantidade").cast("double"))
    .withColumn("valor_total", F.col("valor_total").cast("double"))
)

StatementMeta(, d7e4f1a3-25db-4950-a241-8f56f7ba241c, 6, Finished, Available, Finished, False)

In [5]:
# 3) Last trading day in trades table
last_trade_day = trades_typed.select(F.max("trade_date").alias("d")).first()["d"]

last_day_trades = trades_typed.filter(F.col("trade_date") == F.lit(last_trade_day))

StatementMeta(, d7e4f1a3-25db-4950-a241-8f56f7ba241c, 7, Finished, Available, Finished, False)

In [6]:
# 4) Build deltas by trader+ticker (buy positive, sell negative)
buy_deltas = (
    last_day_trades
    .select(
        F.col("comprador_id").alias("trader_id"),
        "ticker",
        F.col("quantidade").alias("delta_qty"),
        F.col("valor_total").alias("delta_cost")
    )
)

sell_deltas = (
    last_day_trades
    .select(
        F.col("vendedor_id").alias("trader_id"),
        "ticker",
        (-F.col("quantidade")).alias("delta_qty"),
        (-F.col("valor_total")).alias("delta_cost")
    )
)

deltas = (
    buy_deltas.unionByName(sell_deltas)
    .groupBy("trader_id", "ticker")
    .agg(
        F.sum("delta_qty").alias("delta_qty"),
        F.sum("delta_cost").alias("delta_cost")
    )
)

StatementMeta(, d7e4f1a3-25db-4950-a241-8f56f7ba241c, 8, Finished, Available, Finished, False)

In [8]:
# 5) Get latest snapshot row per trader+ticker from positions
w = Window.partitionBy("trader_id", "ticker").orderBy(F.col("position_date").desc())

latest_positions = (
    positions_typed
    .withColumn("rn", F.row_number().over(w))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

StatementMeta(, d7e4f1a3-25db-4950-a241-8f56f7ba241c, 10, Finished, Available, Finished, False)

In [9]:
# 6) Apply deltas to positions
updated_positions = (
    latest_positions.alias("p")
    .join(deltas.alias("d"), on=["trader_id", "ticker"], how="full_outer")
    .select(
        F.lit(last_trade_day).alias("position_date"),
        F.coalesce(F.col("p.trader_id"), F.col("d.trader_id")).alias("trader_id"),
        F.coalesce(F.col("p.ticker"), F.col("d.ticker")).alias("ticker"),
        (F.coalesce(F.col("p.quantity"), F.lit(0.0)) + F.coalesce(F.col("d.delta_qty"), F.lit(0.0))).alias("quantity"),
        (F.coalesce(F.col("p.cost_basis_total"), F.lit(0.0)) + F.coalesce(F.col("d.delta_cost"), F.lit(0.0))).alias("cost_basis_total")
    )
    .withColumn(
        "avg_price",
        F.when(F.col("quantity") != 0, F.col("cost_basis_total") / F.col("quantity")).otherwise(F.lit(0.0))
    )
    .select("position_date", "trader_id", "ticker", "quantity", "avg_price", "cost_basis_total")
)

StatementMeta(, d7e4f1a3-25db-4950-a241-8f56f7ba241c, 11, Finished, Available, Finished, False)

In [10]:
# drop fully closed positions
updated_positions = updated_positions.filter(F.col("quantity") != 0)

StatementMeta(, d7e4f1a3-25db-4950-a241-8f56f7ba241c, 12, Finished, Available, Finished, False)

In [11]:
# 7) Save as updated table
updated_positions.write.mode("overwrite").saveAsTable("dbo.updated_positions")

StatementMeta(, d7e4f1a3-25db-4950-a241-8f56f7ba241c, 13, Finished, Available, Finished, False)